# SEC Filings Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kineviz/fortune500/blob/main/pipeline.ipynb)

Full pipeline from scraping 10-K filings through BigQuery AI extraction and property graph creation.

## Pipeline Steps
1. **Google Cloud Authentication** – Authenticate and set active GCP project
2. **Scraper** – Download 10-K filings from SEC.gov
3. **Parser** – Parse SGML to Markdown (optional, for inspection)
4. **Extract Sections** – Parse SGML to JSONL (Item 1, 1A, 7)
5. **Upload JSONL to GCS** – Sync extracted sections to Cloud Storage
6. **Initialize BigQuery + Load Sections** – Create core tables and load sections JSONL
7. **Run BigQuery Extraction** – Generate `insights` via `AI.GENERATE_TEXT`
8. **Build graph tables in BigQuery** – Parse `insights` and run SQL + `AI.GENERATE_TEXT` normalization
9. **Create Property Graph DDL** – Build `SecGraph` in BigQuery


### 0.0 GCP Environment Setup Checklist
Before running this notebook, ensure your Google Cloud environment is fully configured:

1.  **API Enablement**: Enable [BigQuery API](https://console.cloud.google.com/marketplace/product/google/bigquery.googleapis.com) and [Vertex AI API](https://console.cloud.google.com/marketplace/product/google/aiplatform.googleapis.com).
2.  **Billing**: Ensure your project is linked to an active **[Billing Account](https://console.cloud.google.com/billing/enable)**.
3.  **BigQuery AI Connection**:
    - Create a Cloud Resource Connection named **`vertex_ai_connection`** in the **US** location.
    - Grant the connection's Service Account the **`roles/aiplatform.user`** role.
    - **Run the cell below** to automate this setup.

## Configuration

Set the environment constants below. These variables control GCP authentication, local and cloud storage, and pipeline depth.

### GCP Settings
- **`GCP_PROJECT`**: Your Google Cloud Project ID. Find this in the [GCP Console Project Dashboard](https://console.cloud.google.com/home/dashboard). Ensure you have the [BigQuery API](https://console.cloud.google.com/marketplace/product/google/bigquery.googleapis.com) and [Vertex AI API](https://console.cloud.google.com/marketplace/product/google/aiplatform.googleapis.com) enabled.
- **`GCS_BUCKET`**: The Google Cloud Storage bucket used for staging data for BigQuery load. Format: `gs://your-bucket-name`.
- **`BQ_LOCATION`**: The regional location for your BigQuery datasets (e.g., `US`, `EU`).
- **`BQ_DATASET`**: BigQuery dataset name (default matches `00_run_full_pipeline.sh`: `fortune500_test`).
- **`GEMINI_MODEL`**: Vertex model id for the remote BigQuery model (e.g. `gemini-3.1-pro-preview`).
- **`FORCE_FULL_INSIGHTS_REFRESH`**: If `True`, drops `insights` before init so extraction refills from scratch (same as the shell script).
- **`BQ_MODEL`**: Unused by SQL; the created model object is named `gemini_pro_latest` in `04_init_tables.sql`.
- **Billing**: BigQuery AI functions require an active billing account. Ensure it is enabled [here](https://console.cloud.google.com/billing/enable).

### Local Directories
These variables define where data is stored on your local machine or external drive.
- **`SGML_DIR`**: Root directory for raw `.sgml` files from SEC.gov.
- **`MARKDOWN_DIR`**: Directory for converted `.md` files.
- **`JSON_DIR`**: Directory for sectioned JSONL files (Item 1, 1A, 7).

### Scraper Settings
- **`TICKERS`**: Optional list (e.g. `["AAPL","MSFT"]`). When non-empty, section 6.1 scopes the BigQuery load to those tickers’ `sections.jsonl` files (same idea as `./00_run_full_pipeline.sh AAPL,MSFT`).
- **`SCRAPER_LIMIT`**: The number of companies to pull from the companies list (e.g., 5, 20, 500). Use a small number to test.
- **`SCRAPER_LAST_N_YEARS`**: The number of historical years to scrape for each company.

In [ ]:
GCP_PROJECT = "YOUR_PROJECT_ID"  # Change to your project ID
GCS_BUCKET = "gs://YOUR_BUCKET"  # Change to your bucket
BQ_LOCATION = "US"
BQ_DATASET = "sec_filings"  # Default matches 00_run_full_pipeline.sh (BQ_DATASET)
GEMINI_MODEL = "gemini-3.1-pro-preview"
FORCE_FULL_INSIGHTS_REFRESH = False  # True: DROP insights before init (full re-extract)
BQ_MODEL = "gemini_pro_3"

# Local data directories
SGML_DIR = "../data/sgml"
MARKDOWN_DIR = "../data/markdown"
JSON_DIR = "../data/json"

# Optional explicit ticker list. Example: ["GOOGL", "AAPL", "MSFT"]
TICKERS = []

# Pipeline depth (used when TICKERS is empty)
SCRAPER_LIMIT = 2
SCRAPER_LAST_N_YEARS = 2
SCRAPER_OUTPUT = SGML_DIR

# Derived project constants
BQ_PROJECT = GCP_PROJECT

In [ ]:
# Configure these for your environment
import os
# Assert configuration is updated
from IPython import get_ipython

# Default to invalid until checks pass
_CONFIG_VALIDATED = False

def _block_until_config_validated(_info):
    if not globals().get("_CONFIG_VALIDATED", False):
        raise RuntimeError(
            "Configuration is invalid. Re-run the Configuration cell and fix all errors before running other cells."
        )

ip = get_ipython()
if ip is not None:
    # Register once per kernel session
    if not globals().get("_CONFIG_GUARD_REGISTERED", False):
        ip.events.register("pre_run_cell", _block_until_config_validated)
        _CONFIG_GUARD_REGISTERED = True

errors = []
if GCP_PROJECT == "YOUR_PROJECT_ID":
    errors.append("Update GCP_PROJECT at the top of the notebook.")
if GCS_BUCKET == "gs://YOUR_BUCKET":
    errors.append("Update GCS_BUCKET at the top of the notebook.")
if "-" in BQ_DATASET:
    errors.append("BQ_DATASET cannot contain '-' (use underscores instead).")

if errors:
    raise RuntimeError("Configuration validation failed:\n- " + "\n- ".join(errors))

_CONFIG_VALIDATED = True
print("✓ Configuration validated")


## 0. Colab Setup
If you are running this in Google Colab, you need to clone the repository and install requirements to access the custom scraper scripts and SQL files.

In [ ]:
import sys
import os
from IPython import get_ipython
ipy = get_ipython()

if "google.colab" in sys.modules:
    def has_repo(path):
        return os.path.exists(os.path.join(path, "01_scraper.py"))

    cwd = os.getcwd()
    candidates = [
        cwd,
        os.path.join(cwd, "fortune500"),
        "/content/fortune500",
        "/content/fortune500/fortune500",
    ]
    repo_root = next((p for p in candidates if has_repo(p)), None)

    if repo_root is None:
        if not os.path.exists("/content/fortune500"):
            ipy.system("git clone https://github.com/Kineviz/fortune500.git /content/fortune500")
        repo_root = "/content/fortune500"

    os.chdir(repo_root)
    print(f"Colab repo root: {os.getcwd()}")

ipy.system("pip install -r requirements.txt")


## 1. Setup & Google Cloud Authentication

In [ ]:
# Using GCP_PROJECT from configuration above

if GCP_PROJECT != "YOUR_PROJECT_ID":
    import sys
    from IPython import get_ipython
    import subprocess
    import google.auth
    ipy = get_ipython()
    ipy.system(f"gcloud config set project {GCP_PROJECT}")

    # In Colab, this sets up Application Default Credentials for Python clients
    if 'google.colab' in sys.modules:
        from google.colab import auth as colab_auth
        colab_auth.authenticate_user()
    else:
        ipy.system("gcloud auth login")
        ipy.system("gcloud auth application-default login")

    result = subprocess.run(["gcloud", "config", "get-value", "project"], capture_output=True, text=True)
    current = (result.stdout or "").strip() or "(unset)"
    print(f"✓ Project verified: {current}") if current == GCP_PROJECT else print(f"⚠ Current: {current}")

    # Verify ADC is available to BigQuery Python client
    creds, adc_project = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
    print(f"✓ ADC ready (quota project: {getattr(creds, 'quota_project_id', None) or adc_project or GCP_PROJECT})")
else:
    raise ValueError("Set GCP_PROJECT in the Configuration cell at the beginning of the notebook.")

In [ ]:
import subprocess
import os
import sys
import json

def check_billing(project_id):
    print(f"Checking billing status for {project_id}...")
    try:
        # Using gcloud to verify if billing is enabled on the given project
        result = subprocess.run(
            ["gcloud", "billing", "projects", "describe", project_id, "--format=json"],
            capture_output=True, text=True, check=True
        )
        billing_info = json.loads(result.stdout)
        if billing_info.get("billingEnabled"):
            print("\u2705 Billing is enabled! You are ready to use BigQuery AI.")
        else:
            print("\u274c ERROR: Billing is NOT enabled for this project.")
            print("BigQuery AI will not function until billing is enabled in the Google Cloud Console.")
            print(f"Visit: https://console.cloud.google.com/billing/enable?project={project_id}")
            raise Exception("Billing not enabled")
    except subprocess.CalledProcessError as e:
        print("\u26a0 Could not verify billing status automatically. Output:")
        print(e.stderr)
        print("\nEnsure you are authenticated and have permissions (roles/billing.projectManager).")

if GCP_PROJECT and GCP_PROJECT != "YOUR_PROJECT_ID":
    check_billing(GCP_PROJECT)


### Ensure we're in the project root (where 01_scraper.py, SQL files, list.csv live)


In [ ]:
import os

def find_project_root():
    cwd = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(cwd, "01_scraper.py")):
            return cwd
        parent = os.path.dirname(cwd)
        if parent == cwd:
            break
        cwd = parent
    return os.getcwd()

root = find_project_root()
os.chdir(root)
print(f"Working directory: {os.getcwd()}")

# Create local data directories if they don't exist
for d in [SGML_DIR, MARKDOWN_DIR, JSON_DIR]:
    os.makedirs(d, exist_ok=True)
print(f"Verified directories: {SGML_DIR}, {MARKDOWN_DIR}, {JSON_DIR}")

### Verify GCS bucket exists and is accessible


In [ ]:
import subprocess
import os
import sys

result = subprocess.run(
    ["gsutil", "ls", GCS_BUCKET],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print(f"✓ Bucket accessible: {GCS_BUCKET}")
else:
    print(f"Bucket not found. Creating {GCS_BUCKET}...")
    create = subprocess.run(["gsutil", "mb", "-l", BQ_LOCATION, GCS_BUCKET], capture_output=True, text=True)
    if create.returncode == 0:
        print(f"✓ Bucket created: {GCS_BUCKET}")
    else:
        print(f"⚠ Failed to create bucket: {create.stderr or create.stdout}")

In [ ]:
import asyncio
import sys

# Add project root to path
sys.path.insert(0, os.getcwd())

from importlib.util import spec_from_file_location, module_from_spec

# Load scraper module
spec = spec_from_file_location("scraper", "01_scraper.py")
scraper_mod = module_from_spec(spec)
spec.loader.exec_module(scraper_mod)

## 2. Run Scraper (01_scraper.py)

In [ ]:
# Run scraper for explicit tickers (if provided), otherwise use list.csv + SCRAPER_LIMIT
tickers = [t.strip().upper() for t in (TICKERS or []) if str(t).strip()]

if tickers:
    print(f"Scraping explicit tickers: {tickers} ({SCRAPER_LAST_N_YEARS} years each)...")
    for t in tickers:
        scraper = scraper_mod.SECScraper(
            ticker=t,
            last_n_years=SCRAPER_LAST_N_YEARS,
            workers=5,
            output_dir=SCRAPER_OUTPUT,
            dry_run=False,
        )
        await scraper.run()  # Jupyter has its own event loop
else:
    scraper = scraper_mod.SECScraper(
        limit=SCRAPER_LIMIT,
        last_n_years=SCRAPER_LAST_N_YEARS,
        workers=5,
        output_dir=SCRAPER_OUTPUT,
        dry_run=False,
    )

    # Scraper has internal tqdm; get expected count for context
    import pandas as pd
    try:
        n_companies = min(SCRAPER_LIMIT, len(pd.read_csv("list.csv")))
        print(f"Scraping up to {n_companies} companies ({SCRAPER_LAST_N_YEARS} years each)...")
    except FileNotFoundError:
        n_companies = SCRAPER_LIMIT
    await scraper.run()  # Jupyter has its own event loop


In [ ]:
import subprocess
import os
import sys
from tqdm.auto import tqdm

## 3. Run Parser (02_parser.py) (Optional)

In [ ]:
# Count SGML filings to process
n_sgml = sum(1 for r, d, f in os.walk(SGML_DIR) if "full-submission.txt" in f)
with tqdm(total=max(1, n_sgml), desc="Parsing SGML to Markdown", unit="filing") as pbar:
    subprocess.run(
        [sys.executable, "02_parser.py", "--input_base", SGML_DIR, "--output_base", MARKDOWN_DIR],
        check=True,
        cwd=os.getcwd(),
    )
    pbar.update(max(1, n_sgml))

## 4. Extract Sections (03_extract_sections.py)

In [ ]:
import subprocess
import os
import sys
from tqdm.auto import tqdm

# Count SGML filings to process
n_sgml = sum(1 for r, d, f in os.walk(SGML_DIR) if "full-submission.txt" in f)
with tqdm(total=max(1, n_sgml), desc="Extracting sections to JSONL", unit="section") as pbar:
    subprocess.run(
        [sys.executable, "03_extract_sections.py", "--input_base", SGML_DIR, "--output_base", JSON_DIR],
        check=True,
        cwd=os.getcwd(),
    )
    pbar.update(max(1, n_sgml))

## 5. Upload JSONL to GCS

In [ ]:
import os
import glob
from IPython import get_ipython
from tqdm.auto import tqdm
json_src = os.path.abspath(JSON_DIR)
os.makedirs(json_src, exist_ok=True)
json_files = glob.glob(os.path.join(json_src, "*", "*", "sections.jsonl"))
n_json = len(json_files)
if n_json == 0:
    print(f"⚠ {json_src} is empty. Run Steps 2–4 (scraper, parser, extract sections) first, then re-run this cell.")
else:
    with tqdm(total=n_json, desc="Uploading JSONL to GCS", unit="file") as pbar:
        get_ipython().system(f'gsutil -m rsync -r {json_src}/ {GCS_BUCKET}/json')
        pbar.update(n_json)
    print(f"✓ Synced {n_json} section files to {GCS_BUCKET}/json")

## 6. BigQuery extraction & graph build

Run `AI.GENERATE_TEXT` extraction in BigQuery, then materialize node/edge tables with SQL and optional follow-on `AI.GENERATE_TEXT` normalization—entirely inside BigQuery.


### 6.1 Initialize Tables and Load Sections

Matches `00_run_full_pipeline.sh` step 5: optional `FORCE_FULL_INSIGHTS_REFRESH`, `04_init_tables.sql`, then append-load `sections` from GCS URIs derived from local `JSON_DIR` (optionally filtered by `TICKERS`).

In [ ]:
import os
import subprocess
import google.auth
from google.cloud import bigquery
from pathlib import Path

if 'google.colab' in __import__('sys').modules:
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user()

os.environ.setdefault("GEMINI_MODEL", GEMINI_MODEL)

creds, adc_project = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
client = bigquery.Client(project=GCP_PROJECT, credentials=creds)
print(f"✓ BigQuery client ready (ADC project: {adc_project or GCP_PROJECT})")

def bq_gemini_endpoint(model_env: str, project_id: str) -> str:
    m = (model_env or "gemini-2.5-pro").strip()
    if m.startswith("http://") or m.startswith("https://"):
        return m
    if m.startswith("models/"):
        m = m[len("models/") :].lstrip()
    return (
        f"https://aiplatform.googleapis.com/v1/projects/{project_id}/"
        f"locations/global/publishers/google/models/{m}"
    )

def run_bq_query(filename):
    with open(filename, 'r') as f:
        sql = f.read()
    sql = sql.replace('sec_filings.', f"{BQ_DATASET}.")
    sql = sql.replace('sec_filings;', f"{BQ_DATASET};")
    gemini = os.environ.get("GEMINI_MODEL") or GEMINI_MODEL
    sql = sql.replace("__GEMINI_ENDPOINT__", bq_gemini_endpoint(gemini, GCP_PROJECT))
    print(f"Executing {filename}...")
    job = client.query(sql, location=BQ_LOCATION)
    job.result()
    print(f"✓ Executed {filename}")

_fr = os.environ.get("FORCE_FULL_INSIGHTS_REFRESH")
if _fr is None:
    _fr = globals().get("FORCE_FULL_INSIGHTS_REFRESH", False)
_wipe = str(_fr).strip().lower() in ("1", "true", "yes", "y") or _fr is True
if _wipe:
    try:
        client.query(
            f"DROP TABLE IF EXISTS `{GCP_PROJECT}.{BQ_DATASET}.insights`",
            location=BQ_LOCATION,
        ).result()
        print("✓ Dropped insights table (FORCE_FULL_INSIGHTS_REFRESH)")
    except Exception as e:
        print(f"ℹ Skipped insights drop: {e}")
else:
    print("ℹ Keeping existing insights table (incremental extraction)")

run_bq_query("04_init_tables.sql")

print("Loading sections into BigQuery...")
sections_schema = "filing_id:STRING,company:STRING,company_name:STRING,cik:STRING,sic:STRING,irs_number:STRING,state_of_inc:STRING,org_name:STRING,sec_file_number:STRING,film_number:STRING,business_street_1:STRING,business_street_2:STRING,business_city:STRING,business_state:STRING,business_zip:STRING,business_phone:STRING,mail_street_1:STRING,mail_street_2:STRING,mail_city:STRING,mail_state:STRING,mail_zip:STRING,filing_url:STRING,year:INTEGER,section_id:STRING,content:STRING"

local_json_dir = Path(JSON_DIR)
local_sections = sorted(local_json_dir.glob("*/*/sections.jsonl"))
_tickers = globals().get("TICKERS") or []
if _tickers:
    ticker_set = {str(t).strip().upper() for t in _tickers if str(t).strip()}
    local_sections = [p for p in local_sections if p.parent.parent.name.upper() in ticker_set]

if local_sections:
    section_uris = [
        f"{GCS_BUCKET}/json/{p.parent.parent.name}/{p.parent.name}/sections.jsonl"
        for p in local_sections
    ]
    print(f"✓ Scoped sections load to {len(section_uris)} file(s)")
else:
    print("⚠ No local sections.jsonl found; loading entire GCS prefix via wildcard")
    section_uris = [f"{GCS_BUCKET}/json/*/*/sections.jsonl"]

type_map = {"STRING": "STRING", "INTEGER": "INT64"}
schema = []
for col in sections_schema.split(','):
    name, typ = col.split(':')
    schema.append(bigquery.SchemaField(name, type_map.get(typ, typ)))

table_id = f"{GCP_PROJECT}.{BQ_DATASET}.sections"
load_config = bigquery.LoadJobConfig(
    schema=schema,
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
)

load_job = client.load_table_from_uri(section_uris, table_id, job_config=load_config, location=BQ_LOCATION)
load_job.result()
print("✓ Sections loaded")


### 6.2 Run AI Extraction in BigQuery

Matches the shell script steps 6–7a: align `insights` schema to the remote model, run `05_extraction.sql`, then normalize `ml_generate_text_*` / `result` column names for the graph build.

In [ ]:
from google.api_core import exceptions as gexc

insights_id = f"{GCP_PROJECT}.{BQ_DATASET}.insights"
sections_id = f"{GCP_PROJECT}.{BQ_DATASET}.sections"
model_id = f"{GCP_PROJECT}.{BQ_DATASET}.gemini_pro_latest"


def insights_column_names_from_model():
    q = f"""
    SELECT * FROM AI.GENERATE_TEXT(
      MODEL `{model_id}`,
      (SELECT 'dummy' AS prompt, * FROM `{sections_id}` LIMIT 0),
      STRUCT(0.2 AS temperature, 8192 AS max_output_tokens)
    )
    LIMIT 0
    """
    try:
        dry = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
        job = client.query(q, job_config=dry, location=BQ_LOCATION)
        job.result()
        sch = job.schema
        if sch:
            return tuple(f.name for f in sch)
    except Exception as e:
        print(f"⚠ Dry-run schema probe failed ({e}); running LIMIT 0 query once...")
    try:
        job = client.query(q, location=BQ_LOCATION)
        rows = job.result()
        job.reload()
        sch = job.schema or getattr(rows, "schema", None)
        if sch:
            return tuple(f.name for f in sch)
    except Exception as e:
        print(f"⚠ Could not probe AI.GENERATE_TEXT output schema: {e}")
    return None


def rebuild_insights_empty_canonical():
    q = f"""
    CREATE OR REPLACE TABLE `{insights_id}` AS
    SELECT * FROM AI.GENERATE_TEXT(
      MODEL `{model_id}`,
      (SELECT 'dummy' AS prompt, * FROM `{sections_id}` LIMIT 0),
      STRUCT(0.2 AS temperature, 8192 AS max_output_tokens)
    )
    LIMIT 0
    """
    client.query(q, location=BQ_LOCATION).result()


expected = insights_column_names_from_model()
try:
    cur = client.get_table(insights_id)
    actual = tuple(f.name for f in cur.schema)
except gexc.NotFound:
    actual = None

if expected is not None and actual == expected:
    pass
else:
    if expected is None:
        print("ℹ Rebuilding insights table (could not compare schemas; ensuring INSERT compatibility)")
    else:
        print(
            f"ℹ Aligning insights schema to current model output "
            f"({'missing table' if actual is None else f'{len(actual)} cols'} → {len(expected)} cols)"
        )
    rebuild_insights_empty_canonical()
    print("✓ insights table recreated empty (05_extraction refills from sections per NOT EXISTS)")

run_bq_query("05_extraction.sql")

insights_table = insights_id
t = client.get_table(insights_table)
schema_names = {f.name for f in t.schema}

if "ml_generate_text_result" not in schema_names:
    if "result" in schema_names:
        client.query(
            f"""
            CREATE OR REPLACE TABLE `{insights_table}` AS
            SELECT i.*, CAST(i.result AS STRING) AS ml_generate_text_result
            FROM `{insights_table}` AS i
            """,
            location=BQ_LOCATION,
        ).result()
        print("✓ Added ml_generate_text_result from result column")
    elif "ml_generate_text_llm_result" in schema_names:
        client.query(
            f"""
            CREATE OR REPLACE TABLE `{insights_table}` AS
            SELECT i.*, CAST(i.ml_generate_text_llm_result AS STRING) AS ml_generate_text_result
            FROM `{insights_table}` AS i
            """,
            location=BQ_LOCATION,
        ).result()
        print("✓ Added ml_generate_text_result from ml_generate_text_llm_result column")

t = client.get_table(insights_table)
schema_names = {f.name for f in t.schema}
if "ml_generate_text_llm_result" not in schema_names and "ml_generate_text_result" in schema_names:
    client.query(
        f"""
        CREATE OR REPLACE TABLE `{insights_table}` AS
        SELECT i.*, CAST(i.ml_generate_text_result AS STRING) AS ml_generate_text_llm_result
        FROM `{insights_table}` AS i
        """,
        location=BQ_LOCATION,
    ).result()
    print("✓ Added ml_generate_text_llm_result mirror column")

print("✓ Extraction and insights column alignment complete")


### 6.3 Build graph tables in BigQuery (BigQuery-only normalization)

Materialize every property-graph node and edge table directly in BigQuery. Raw `insights` JSON is parsed in SQL (including stripping optional ``` fences). Distinct competitor, risk, and market labels are classified with `AI.GENERATE_TEXT` using the same remote model object as extraction (`gemini_pro_latest` from `04_init_tables.sql`).


In [ ]:
# Build all property-graph input tables in BigQuery (SQL + AI.GENERATE_TEXT only).
# Reads `insights` from step 6.2. Uses AI.GENERATE_TEXT for normalization (competitors, risks, markets).

P, D = GCP_PROJECT, BQ_DATASET
# Remote model object from 04_init_tables.sql (override if you use a different MODEL name).
ML_MODEL = f"`{P}.{D}.gemini_pro_latest`"


def _ex(sql: str) -> None:
    job = client.query(sql, location=BQ_LOCATION)
    job.result()


def _scalar(q: str):
    return list(client.query(q, location=BQ_LOCATION).result())[0][0]


# --- 1) Parse insights JSON (strip ``` fences; fallback = substring from first { to last }) ---
_ex(
    f"""
CREATE OR REPLACE TABLE `{P}.{D}.graph_staging_insights_json` AS
WITH raw AS (
  SELECT
    i.*,
    TRIM(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          COALESCE(
            JSON_VALUE(TO_JSON_STRING(i), '$.ml_generate_text_result'),
            JSON_VALUE(TO_JSON_STRING(i), '$.ml_generate_text_llm_result'),
            JSON_VALUE(TO_JSON_STRING(i), '$.result'),
            ''
          ),
          r'(?s)^```[a-zA-Z0-9]*\\s*',
          ''
        ),
        r'\\s*```$',
        ''
      )
    ) AS cleaned_llm
  FROM `{P}.{D}.insights` AS i
)
SELECT
  *,
  COALESCE(
    SAFE.PARSE_JSON(cleaned_llm),
    IF(
      STRPOS(cleaned_llm, '{{') > 0,
      SAFE.PARSE_JSON(
        SUBSTR(
          cleaned_llm,
          STRPOS(cleaned_llm, '{{'),
          LENGTH(cleaned_llm) - STRPOS(REVERSE(cleaned_llm), '}}') - STRPOS(cleaned_llm, '{{') + 2
        )
      ),
      NULL
    )
  ) AS j
FROM raw;
"""
)

# --- 2) Company / document / section ---
_ex(
    f"""
CREATE OR REPLACE TABLE `{P}.{D}.nodes_company` AS
SELECT
  company AS id,
  ANY_VALUE(company_name) AS label,
  CAST(ANY_VALUE(cik) AS STRING) AS cik,
  CAST(ANY_VALUE(sic) AS STRING) AS sic,
  CAST(ANY_VALUE(irs_number) AS STRING) AS irs_number,
  ANY_VALUE(state_of_inc) AS state_of_inc,
  ANY_VALUE(org_name) AS org_name,
  CAST(ANY_VALUE(sec_file_number) AS STRING) AS sec_file_number,
  CAST(ANY_VALUE(film_number) AS STRING) AS film_number,
  ANY_VALUE(business_street_1) AS business_street_1,
  ANY_VALUE(business_street_2) AS business_street_2,
  ANY_VALUE(business_city) AS business_city,
  ANY_VALUE(business_state) AS business_state,
  CAST(ANY_VALUE(business_zip) AS STRING) AS business_zip,
  CAST(ANY_VALUE(business_phone) AS STRING) AS business_phone,
  ANY_VALUE(mail_street_1) AS mail_street_1,
  ANY_VALUE(mail_street_2) AS mail_street_2,
  ANY_VALUE(mail_city) AS mail_city,
  ANY_VALUE(mail_state) AS mail_state,
  CAST(ANY_VALUE(mail_zip) AS STRING) AS mail_zip
FROM `{P}.{D}.graph_staging_insights_json`
WHERE company IS NOT NULL AND company != ''
GROUP BY company;

CREATE OR REPLACE TABLE `{P}.{D}.nodes_document` AS
SELECT DISTINCT
  filing_url AS id,
  year,
  CAST(sec_file_number AS STRING) AS sec_file_number,
  CAST(film_number AS STRING) AS film_number,
  filing_url AS link,
  company,
  company_name,
  CAST(cik AS STRING) AS cik
FROM `{P}.{D}.graph_staging_insights_json`
WHERE filing_url IS NOT NULL AND filing_url != '';

CREATE OR REPLACE TABLE `{P}.{D}.nodes_section` AS
SELECT DISTINCT
  CONCAT(filing_url, '#', section_id) AS id,
  section_id AS label,
  section_id AS section,
  filing_url AS document_id,
  year,
  company,
  company_name,
  CONCAT(filing_url, '#', section_id) AS link
FROM `{P}.{D}.graph_staging_insights_json`
WHERE filing_url IS NOT NULL AND filing_url != '' AND section_id IS NOT NULL;
"""
)

# --- 3) Staging rows with UUIDs per extracted entity ---
_ex(
    f"""
CREATE OR REPLACE TABLE `{P}.{D}.graph_staging_market_rows` AS
SELECT
  GENERATE_UUID() AS id,
  company,
  year,
  section_id,
  filing_url,
  company_name,
  JSON_VALUE(market_item, '$.market') AS label,
  COALESCE(JSON_VALUE(market_item, '$.evidence'), JSON_VALUE(market_item, '$.details'), '') AS evidence,
  market_action,
  JSON_VALUE(market_item, '$.reference') AS ref_text
FROM (
  SELECT i.company, i.year, i.section_id, i.filing_url, i.company_name, i.j, 'Entering' AS market_action, m AS market_item
  FROM `{P}.{D}.graph_staging_insights_json` AS i
  LEFT JOIN UNNEST(
    IF(JSON_TYPE(JSON_QUERY(i.j, '$.markets')) = 'object', JSON_QUERY_ARRAY(i.j, '$.markets.entering'), ARRAY<JSON>[])
  ) AS m ON TRUE
  UNION ALL
  SELECT i.company, i.year, i.section_id, i.filing_url, i.company_name, i.j, 'Exiting' AS market_action, m AS market_item
  FROM `{P}.{D}.graph_staging_insights_json` AS i
  LEFT JOIN UNNEST(
    IF(JSON_TYPE(JSON_QUERY(i.j, '$.markets')) = 'object', JSON_QUERY_ARRAY(i.j, '$.markets.exiting'), ARRAY<JSON>[])
  ) AS m ON TRUE
  UNION ALL
  SELECT i.company, i.year, i.section_id, i.filing_url, i.company_name, i.j, 'Expanding' AS market_action, m AS market_item
  FROM `{P}.{D}.graph_staging_insights_json` AS i
  LEFT JOIN UNNEST(
    IF(JSON_TYPE(JSON_QUERY(i.j, '$.markets')) = 'object', JSON_QUERY_ARRAY(i.j, '$.markets.expanding'), ARRAY<JSON>[])
  ) AS m ON TRUE
) AS x
WHERE j IS NOT NULL
  AND JSON_VALUE(market_item, '$.market') IS NOT NULL
  AND TRIM(JSON_VALUE(market_item, '$.market')) != '';

CREATE OR REPLACE TABLE `{P}.{D}.graph_staging_risk_rows` AS
SELECT
  GENERATE_UUID() AS id,
  company,
  year,
  section_id,
  filing_url,
  company_name,
  JSON_VALUE(r, '$.risk') AS label,
  COALESCE(JSON_VALUE(r, '$.description'), '') AS description,
  JSON_VALUE(r, '$.reference') AS ref_text
FROM `{P}.{D}.graph_staging_insights_json`
LEFT JOIN UNNEST(
  IF(
    JSON_TYPE(JSON_QUERY(j, '$.risks_opportunities')) = 'object',
    JSON_QUERY_ARRAY(j, '$.risks_opportunities.emerging_risks'),
    ARRAY<JSON>[]
  )
) AS r
ON TRUE
WHERE j IS NOT NULL
  AND JSON_VALUE(r, '$.risk') IS NOT NULL
  AND TRIM(JSON_VALUE(r, '$.risk')) != '';

CREATE OR REPLACE TABLE `{P}.{D}.graph_staging_opp_rows` AS
SELECT
  GENERATE_UUID() AS id,
  company,
  year,
  section_id,
  filing_url,
  company_name,
  JSON_VALUE(o, '$.opportunity') AS label,
  COALESCE(JSON_VALUE(o, '$.description'), '') AS description,
  JSON_VALUE(o, '$.reference') AS ref_text
FROM `{P}.{D}.graph_staging_insights_json`
LEFT JOIN UNNEST(
  IF(
    JSON_TYPE(JSON_QUERY(j, '$.risks_opportunities')) = 'object',
    JSON_QUERY_ARRAY(j, '$.risks_opportunities.emerging_opportunities'),
    ARRAY<JSON>[]
  )
) AS o
ON TRUE
WHERE j IS NOT NULL
  AND JSON_VALUE(o, '$.opportunity') IS NOT NULL
  AND TRIM(JSON_VALUE(o, '$.opportunity')) != '';

CREATE OR REPLACE TABLE `{P}.{D}.graph_staging_comp_rows` AS
SELECT
  GENERATE_UUID() AS id,
  company,
  year,
  section_id,
  filing_url,
  company_name,
  JSON_VALUE(c, '$.name') AS label,
  COALESCE(JSON_VALUE(c, '$.relationship'), '') AS relationship,
  JSON_VALUE(c, '$.reference') AS ref_text
FROM `{P}.{D}.graph_staging_insights_json`
LEFT JOIN UNNEST(
  IF(JSON_TYPE(j) = 'object', JSON_QUERY_ARRAY(j, '$.competitors'), ARRAY<JSON>[])
) AS c
ON TRUE
WHERE j IS NOT NULL
  AND JSON_VALUE(c, '$.name') IS NOT NULL
  AND TRIM(JSON_VALUE(c, '$.name')) != '';
"""
)

# --- 4) Entity nodes, references, base edges ---
_ex(
    f'''
-- Reference link fragments: encodeURIComponent-style; long refs use start,end pair
CREATE TEMP FUNCTION text_fragment_encode(ref STRING)
RETURNS STRING
LANGUAGE js AS """
function enc(s) {{
  if (s === null || s === undefined) return '';
  return encodeURIComponent(s);
}}
var t = (ref || '').trim();
if (!t) return '';
var w = t.trim().split(/ +/).filter(function(x) {{ return x.length > 0; }});
if (w.length <= 10) return enc(t);
return enc(w.slice(0, 5).join(' ')) + ',' + enc(w.slice(-5).join(' '));
""";

CREATE OR REPLACE TABLE `{P}.{D}.nodes_reference` AS
SELECT * EXCEPT (rn) FROM (
  SELECT
    TO_HEX(MD5(ref_text)) AS id,
    ref_text AS text,
    CONCAT(REPLACE(filing_url, 'ix?doc=/', ''), '#:~:text=', text_fragment_encode(ref_text)) AS link,
    year,
    section_id AS section,
    company,
    company_name,
    ROW_NUMBER() OVER (PARTITION BY TO_HEX(MD5(ref_text)) ORDER BY filing_url, section_id) AS rn
  FROM (
    SELECT filing_url, year, section_id, company, company_name, ref_text
    FROM `{P}.{D}.graph_staging_market_rows` WHERE ref_text IS NOT NULL AND TRIM(ref_text) != ''
    UNION ALL
    SELECT filing_url, year, section_id, company, company_name, ref_text
    FROM `{P}.{D}.graph_staging_risk_rows` WHERE ref_text IS NOT NULL AND TRIM(ref_text) != ''
    UNION ALL
    SELECT filing_url, year, section_id, company, company_name, ref_text
    FROM `{P}.{D}.graph_staging_opp_rows` WHERE ref_text IS NOT NULL AND TRIM(ref_text) != ''
    UNION ALL
    SELECT filing_url, year, section_id, company, company_name, ref_text
    FROM `{P}.{D}.graph_staging_comp_rows` WHERE ref_text IS NOT NULL AND TRIM(ref_text) != ''
  )
) WHERE rn = 1;

CREATE OR REPLACE TABLE `{P}.{D}.nodes_market` AS
SELECT id, label, year, section_id AS section, filing_url AS link, evidence, market_action
FROM `{P}.{D}.graph_staging_market_rows`;

CREATE OR REPLACE TABLE `{P}.{D}.nodes_risk` AS
SELECT id, label, year, section_id AS section, filing_url AS link, description
FROM `{P}.{D}.graph_staging_risk_rows`;

CREATE OR REPLACE TABLE `{P}.{D}.nodes_opportunity` AS
SELECT id, label, year, section_id AS section, filing_url AS link, description
FROM `{P}.{D}.graph_staging_opp_rows`;

CREATE OR REPLACE TABLE `{P}.{D}.nodes_competitor` AS
SELECT id, label, year, section_id AS section, filing_url AS link, relationship
FROM `{P}.{D}.graph_staging_comp_rows`;

CREATE OR REPLACE TABLE `{P}.{D}.edges_filed` AS
SELECT DISTINCT company AS source_node, filing_url AS target_node
FROM `{P}.{D}.graph_staging_insights_json`
WHERE company IS NOT NULL AND filing_url IS NOT NULL;

CREATE OR REPLACE TABLE `{P}.{D}.edges_doc_contains_section` AS
SELECT DISTINCT filing_url AS source_node, CONCAT(filing_url, '#', section_id) AS target_node
FROM `{P}.{D}.graph_staging_insights_json`
WHERE filing_url IS NOT NULL AND section_id IS NOT NULL;

CREATE OR REPLACE TABLE `{P}.{D}.edges_section_belongs_to_document` AS
SELECT DISTINCT CONCAT(filing_url, '#', section_id) AS source_node, filing_url AS target_node
FROM `{P}.{D}.graph_staging_insights_json`
WHERE filing_url IS NOT NULL AND section_id IS NOT NULL;

CREATE OR REPLACE TABLE `{P}.{D}.edges_section_contains_ref` AS
SELECT DISTINCT CONCAT(filing_url, '#', section_id) AS source_node, TO_HEX(MD5(ref_text)) AS target_node
FROM (
  SELECT filing_url, section_id, ref_text FROM `{P}.{D}.graph_staging_market_rows`
    WHERE ref_text IS NOT NULL AND TRIM(ref_text) != ''
  UNION ALL
  SELECT filing_url, section_id, ref_text FROM `{P}.{D}.graph_staging_risk_rows`
    WHERE ref_text IS NOT NULL AND TRIM(ref_text) != ''
  UNION ALL
  SELECT filing_url, section_id, ref_text FROM `{P}.{D}.graph_staging_opp_rows`
    WHERE ref_text IS NOT NULL AND TRIM(ref_text) != ''
  UNION ALL
  SELECT filing_url, section_id, ref_text FROM `{P}.{D}.graph_staging_comp_rows`
    WHERE ref_text IS NOT NULL AND TRIM(ref_text) != ''
);

CREATE OR REPLACE TABLE `{P}.{D}.edges_entering` AS
SELECT company AS source_node, id AS target_node
FROM `{P}.{D}.graph_staging_market_rows` WHERE market_action = 'Entering';

CREATE OR REPLACE TABLE `{P}.{D}.edges_exiting` AS
SELECT company AS source_node, id AS target_node
FROM `{P}.{D}.graph_staging_market_rows` WHERE market_action = 'Exiting';

CREATE OR REPLACE TABLE `{P}.{D}.edges_expanding` AS
SELECT company AS source_node, id AS target_node
FROM `{P}.{D}.graph_staging_market_rows` WHERE market_action = 'Expanding';

CREATE OR REPLACE TABLE `{P}.{D}.edges_faces_risk` AS
SELECT company AS source_node, id AS target_node FROM `{P}.{D}.graph_staging_risk_rows`;

CREATE OR REPLACE TABLE `{P}.{D}.edges_pursuing` AS
SELECT company AS source_node, id AS target_node FROM `{P}.{D}.graph_staging_opp_rows`;

CREATE OR REPLACE TABLE `{P}.{D}.edges_competes` AS
SELECT company AS source_node, id AS target_node FROM `{P}.{D}.graph_staging_comp_rows`;

CREATE OR REPLACE TABLE `{P}.{D}.edges_market_has_reference` AS
SELECT id AS source_node, TO_HEX(MD5(ref_text)) AS target_node
FROM `{P}.{D}.graph_staging_market_rows`
WHERE ref_text IS NOT NULL AND TRIM(ref_text) != '';

CREATE OR REPLACE TABLE `{P}.{D}.edges_risk_has_reference` AS
SELECT id AS source_node, TO_HEX(MD5(ref_text)) AS target_node
FROM `{P}.{D}.graph_staging_risk_rows`
WHERE ref_text IS NOT NULL AND TRIM(ref_text) != '';

CREATE OR REPLACE TABLE `{P}.{D}.edges_opportunity_has_reference` AS
SELECT id AS source_node, TO_HEX(MD5(ref_text)) AS target_node
FROM `{P}.{D}.graph_staging_opp_rows`
WHERE ref_text IS NOT NULL AND TRIM(ref_text) != '';

CREATE OR REPLACE TABLE `{P}.{D}.edges_competitor_has_reference` AS
SELECT id AS source_node, TO_HEX(MD5(ref_text)) AS target_node
FROM `{P}.{D}.graph_staging_comp_rows`
WHERE ref_text IS NOT NULL AND TRIM(ref_text) != '';
'''
)

# --- 5) Static taxonomy hubs ---
_ex(
    f"""
CREATE OR REPLACE TABLE `{P}.{D}.nodes_risk_category` AS
SELECT * FROM UNNEST([
  STRUCT('Cybersecurity & Data Privacy' AS id, 'Cybersecurity & Data Privacy' AS label, 'Threats to information systems, data breaches, privacy regulation.' AS description),
  STRUCT('Climate & Environment', 'Climate & Environment', 'Climate change, extreme weather, environmental regulation.'),
  STRUCT('Regulatory & Compliance', 'Regulatory & Compliance', 'Legal, regulatory, and compliance changes.'),
  STRUCT('AI & Emerging Technology', 'AI & Emerging Technology', 'AI, ML, quantum, and technology disruption.'),
  STRUCT('Supply Chain & Operations', 'Supply Chain & Operations', 'Supply chain, logistics, manufacturing, operations.'),
  STRUCT('Macroeconomic & Inflation', 'Macroeconomic & Inflation', 'Inflation, recession, commodity and cost pressures.'),
  STRUCT('Financial & Capital Markets', 'Financial & Capital Markets', 'Interest rates, credit, liquidity, capital markets.'),
  STRUCT('Foreign Currency & Exchange', 'Foreign Currency & Exchange', 'FX exposure and translation.'),
  STRUCT('Geopolitical & Trade', 'Geopolitical & Trade', 'Geopolitics, sanctions, trade restrictions.'),
  STRUCT('Tax & Fiscal Policy', 'Tax & Fiscal Policy', 'Tax law changes and fiscal policy.'),
  STRUCT('Labor & Human Capital', 'Labor & Human Capital', 'Labor, talent, workforce, wages.'),
  STRUCT('Legal & Litigation', 'Legal & Litigation', 'Litigation, IP, legal proceedings.'),
  STRUCT('Healthcare & Pharma', 'Healthcare & Pharma', 'Healthcare policy, pharma, clinical risks.'),
  STRUCT('ESG & Sustainability', 'ESG & Sustainability', 'ESG scrutiny and sustainability.'),
  STRUCT('Competition & Market Position', 'Competition & Market Position', 'Competitive dynamics and market position.'),
  STRUCT('Corporate Strategy & Execution', 'Corporate Strategy & Execution', 'M&A, restructuring, ERP, execution.'),
  STRUCT('Customer & Demand', 'Customer & Demand', 'Customer behavior and demand shifts.')
]);

CREATE OR REPLACE TABLE `{P}.{D}.nodes_geographic_region` AS
SELECT * FROM UNNEST([
  STRUCT('Global' AS id, 'Global' AS label, 'Worldwide or multi-regional.' AS description),
  STRUCT('North America', 'North America', 'US, Canada, Mexico.'),
  STRUCT('Europe', 'Europe', 'European countries and EU.'),
  STRUCT('Asia Pacific', 'Asia Pacific', 'East / South / Southeast Asia, Oceania.'),
  STRUCT('Latin America', 'Latin America', 'Central and South America, Caribbean.'),
  STRUCT('Middle East & Africa', 'Middle East & Africa', 'MENA and Sub-Saharan Africa.'),
  STRUCT('Russia & CIS', 'Russia & CIS', 'Russia, Ukraine, CIS.')
]);

CREATE OR REPLACE TABLE `{P}.{D}.nodes_market_category` AS
SELECT * FROM UNNEST([
  STRUCT('Cloud & Software' AS id, 'Cloud & Software' AS label, 'Cloud, SaaS, enterprise software, AI platforms.' AS description),
  STRUCT('Renewable Energy', 'Renewable Energy', 'Solar, wind, hydrogen, clean energy.'),
  STRUCT('Electric Vehicles & Mobility', 'Electric Vehicles & Mobility', 'EVs, charging, batteries, autonomy.'),
  STRUCT('Oil & Gas', 'Oil & Gas', 'Petroleum upstream/midstream/downstream, LNG.'),
  STRUCT('Healthcare Services', 'Healthcare Services', 'Pharma, devices, care delivery.'),
  STRUCT('Financial Products', 'Financial Products', 'Banking, insurance, payments, fintech.'),
  STRUCT('E-commerce & Digital', 'E-commerce & Digital', 'Online retail and digital commerce.'),
  STRUCT('Telecommunications', 'Telecommunications', 'Telco, broadband, 5G, fiber.'),
  STRUCT('Power & Utilities', 'Power & Utilities', 'Generation, grid, utilities.'),
  STRUCT('Food & Agriculture', 'Food & Agriculture', 'Food production and agriculture.'),
  STRUCT('Construction & Infrastructure', 'Construction & Infrastructure', 'Construction and infrastructure.'),
  STRUCT('Defense & Space', 'Defense & Space', 'Defense and space systems.')
]);
"""
)

# --- 6) AI: competitor labels (distinct) ---
n_comp = _scalar(f"SELECT COUNT(*) FROM `{P}.{D}.nodes_competitor`")
if n_comp > 0:
    _ex(
        f"""
CREATE OR REPLACE TABLE `{P}.{D}.graph_ai_competitor_norm` AS
SELECT * FROM AI.GENERATE_TEXT(
  MODEL {ML_MODEL},
  (
    SELECT DISTINCT
      label,
      CONCAT(
        'Normalize this SEC 10-K competitor label. Return ONLY raw JSON (no markdown) with keys: ',
        'canonical_name (string), competitor_type (Company|Category|Generic), ',
        'sector (string or null), parent (string or null), product_category (string or null). ',
        'sector must be one of: Technology, Financial Services, Healthcare & Pharma, Energy, Retail & E-commerce, Industrial & Manufacturing, Consumer Goods, Automotive & Transportation, Aerospace & Defense, Telecommunications & Media, Utilities, Real Estate, Materials & Chemicals or null. ',
        'product_category must be one of: Cloud & Software, Renewable Energy, Electric Vehicles & Mobility, Oil & Gas, Healthcare Services, Financial Products, E-commerce & Digital, Telecommunications, Power & Utilities, Food & Agriculture, Construction & Infrastructure, Defense & Space or null. ',
        'Label: ', label
      ) AS prompt
    FROM `{P}.{D}.nodes_competitor`
    WHERE label IS NOT NULL AND TRIM(label) != ''
  ),
  STRUCT(0.1 AS temperature, 2048 AS max_output_tokens)
);
"""
    )
else:
    _ex(
        f"""
CREATE OR REPLACE TABLE `{P}.{D}.graph_ai_competitor_norm` AS
SELECT * FROM AI.GENERATE_TEXT(
  MODEL {ML_MODEL},
  (SELECT 'noop' AS prompt WHERE FALSE),
  STRUCT(0.1 AS temperature, 16 AS max_output_tokens)
);
"""
    )

_ex(
    f"""
CREATE OR REPLACE TABLE `{P}.{D}.graph_parsed_competitor` AS
WITH cleaned AS (
  SELECT
    label AS src_label,
    TRIM(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          COALESCE(
            JSON_VALUE(TO_JSON_STRING(t), '$.ml_generate_text_result'),
            JSON_VALUE(TO_JSON_STRING(t), '$.ml_generate_text_llm_result'),
            JSON_VALUE(TO_JSON_STRING(t), '$.result'),
            ''
          ),
          r'(?s)^```[a-zA-Z0-9]*\\s*',
          ''
        ),
        r'\\s*```$',
        ''
      )
    ) AS txt
  FROM `{P}.{D}.graph_ai_competitor_norm` AS t
),
parsed AS (
  SELECT
    src_label,
    COALESCE(
      SAFE.PARSE_JSON(txt),
      IF(
        STRPOS(txt, '{{') > 0,
        SAFE.PARSE_JSON(
          SUBSTR(
            txt,
            STRPOS(txt, '{{'),
            LENGTH(txt) - STRPOS(REVERSE(txt), '}}') - STRPOS(txt, '{{') + 2
          )
        ),
        NULL
      )
    ) AS j
  FROM cleaned
  WHERE txt IS NOT NULL AND TRIM(txt) != ''
)
SELECT
  src_label AS label,
  TRIM(JSON_VALUE(j, '$.canonical_name')) AS canonical_name,
  TO_HEX(MD5(LOWER(TRIM(JSON_VALUE(j, '$.canonical_name'))))) AS norm_id,
  TRIM(JSON_VALUE(j, '$.competitor_type')) AS competitor_type,
  TRIM(JSON_VALUE(j, '$.sector')) AS sector,
  TRIM(JSON_VALUE(j, '$.parent')) AS parent_name,
  TRIM(JSON_VALUE(j, '$.product_category')) AS product_category
FROM parsed
WHERE JSON_VALUE(j, '$.canonical_name') IS NOT NULL
  AND TRIM(JSON_VALUE(j, '$.canonical_name')) != '';

CREATE OR REPLACE TABLE `{P}.{D}.nodes_normalized_competitor` AS
SELECT
  norm_id AS id,
  ANY_VALUE(canonical_name) AS label,
  ANY_VALUE(competitor_type) AS competitor_type,
  ANY_VALUE(sector) AS sector,
  ANY_VALUE(product_category) AS product_category
FROM `{P}.{D}.graph_parsed_competitor`
GROUP BY norm_id;

CREATE OR REPLACE TABLE `{P}.{D}.edges_instance_of` AS
SELECT DISTINCT c.id AS source_node, p.norm_id AS target_node
FROM `{P}.{D}.nodes_competitor` AS c
JOIN `{P}.{D}.graph_parsed_competitor` AS p ON p.label = c.label;

CREATE OR REPLACE TABLE `{P}.{D}.edges_subsidiary_of` AS
SELECT DISTINCT
  child.norm_id AS source_node,
  parent.norm_id AS target_node
FROM `{P}.{D}.graph_parsed_competitor` AS child
JOIN `{P}.{D}.graph_parsed_competitor` AS parent
  ON LOWER(TRIM(child.parent_name)) = LOWER(TRIM(parent.canonical_name))
WHERE child.parent_name IS NOT NULL
  AND TRIM(child.parent_name) != ''
  AND child.norm_id != parent.norm_id;

CREATE OR REPLACE TABLE `{P}.{D}.edges_in_market_category` AS
SELECT DISTINCT n.id AS source_node, m.id AS target_node
FROM `{P}.{D}.nodes_normalized_competitor` AS n
JOIN `{P}.{D}.nodes_market_category` AS m ON m.id = n.product_category
WHERE n.product_category IS NOT NULL AND TRIM(n.product_category) != '';
"""
)

# --- 7) AI: risk labels -> categories ---
n_risk = _scalar(f"SELECT COUNT(*) FROM `{P}.{D}.nodes_risk`")
if n_risk > 0:
    _ex(
        f"""
CREATE OR REPLACE TABLE `{P}.{D}.graph_ai_risk_cat` AS
SELECT * FROM AI.GENERATE_TEXT(
  MODEL {ML_MODEL},
  (
    SELECT DISTINCT
      label,
      CONCAT(
        'Classify this SEC 10-K risk label into 1-3 categories. Return ONLY raw JSON (no markdown): ',
        '{{"label":"<same label>","categories":["Cat1"]}}. ',
        'Use ONLY these exact category names: Cybersecurity & Data Privacy, Climate & Environment, Regulatory & Compliance, AI & Emerging Technology, Supply Chain & Operations, Macroeconomic & Inflation, Financial & Capital Markets, Foreign Currency & Exchange, Geopolitical & Trade, Tax & Fiscal Policy, Labor & Human Capital, Legal & Litigation, Healthcare & Pharma, ESG & Sustainability, Competition & Market Position, Corporate Strategy & Execution, Customer & Demand. Label: ', label
      ) AS prompt
    FROM `{P}.{D}.nodes_risk`
    WHERE label IS NOT NULL AND TRIM(label) != ''
  ),
  STRUCT(0.1 AS temperature, 1024 AS max_output_tokens)
);
"""
    )
else:
    _ex(
        f"""
CREATE OR REPLACE TABLE `{P}.{D}.graph_ai_risk_cat` AS
SELECT * FROM AI.GENERATE_TEXT(
  MODEL {ML_MODEL},
  (SELECT 'noop' AS prompt WHERE FALSE),
  STRUCT(0.1 AS temperature, 16 AS max_output_tokens)
);
"""
    )

_ex(
    f"""
CREATE OR REPLACE TABLE `{P}.{D}.graph_parsed_risk_cat` AS
WITH cleaned AS (
  SELECT
    label AS src_label,
    TRIM(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          COALESCE(
            JSON_VALUE(TO_JSON_STRING(t), '$.ml_generate_text_result'),
            JSON_VALUE(TO_JSON_STRING(t), '$.ml_generate_text_llm_result'),
            JSON_VALUE(TO_JSON_STRING(t), '$.result'),
            ''
          ),
          r'(?s)^```[a-zA-Z0-9]*\\s*',
          ''
        ),
        r'\\s*```$',
        ''
      )
    ) AS txt
  FROM `{P}.{D}.graph_ai_risk_cat` AS t
),
parsed AS (
  SELECT
    src_label,
    COALESCE(
      SAFE.PARSE_JSON(txt),
      IF(
        STRPOS(txt, '{{') > 0,
        SAFE.PARSE_JSON(
          SUBSTR(
            txt,
            STRPOS(txt, '{{'),
            LENGTH(txt) - STRPOS(REVERSE(txt), '}}') - STRPOS(txt, '{{') + 2
          )
        ),
        NULL
      )
    ) AS j
  FROM cleaned
  WHERE txt IS NOT NULL AND TRIM(txt) != ''
)
SELECT
  src_label AS risk_label,
  cat
FROM parsed,
UNNEST(COALESCE(JSON_VALUE_ARRAY(j, '$.categories'), ARRAY<STRING>[])) AS cat
WHERE cat IS NOT NULL AND TRIM(cat) != '';

CREATE OR REPLACE TABLE `{P}.{D}.edges_has_risk_category` AS
SELECT DISTINCT r.id AS source_node, c.id AS target_node
FROM `{P}.{D}.nodes_risk` AS r
JOIN `{P}.{D}.graph_parsed_risk_cat` AS p ON p.risk_label = r.label
JOIN `{P}.{D}.nodes_risk_category` AS c ON c.id = p.cat;
"""
)

# --- 8) AI: market labels -> region + product_category ---
n_mkt = _scalar(f"SELECT COUNT(*) FROM `{P}.{D}.nodes_market`")
if n_mkt > 0:
    _ex(
        f"""
CREATE OR REPLACE TABLE `{P}.{D}.graph_ai_market_dim` AS
SELECT * FROM AI.GENERATE_TEXT(
  MODEL {ML_MODEL},
  (
    SELECT DISTINCT
      label,
      CONCAT(
        'Classify this SEC 10-K market label. Return ONLY raw JSON (no markdown): ',
        '{{"label":"<same>","geographic_region":null,"product_category":null}}. ',
        'geographic_region must be one of: Global, North America, Europe, Asia Pacific, Latin America, Middle East & Africa, Russia & CIS or null. ',
        'product_category must be one of: Cloud & Software, Renewable Energy, Electric Vehicles & Mobility, Oil & Gas, Healthcare Services, Financial Products, E-commerce & Digital, Telecommunications, Power & Utilities, Food & Agriculture, Construction & Infrastructure, Defense & Space or null. ',
        'Label: ', label
      ) AS prompt
    FROM `{P}.{D}.nodes_market`
    WHERE label IS NOT NULL AND TRIM(label) != ''
  ),
  STRUCT(0.1 AS temperature, 1024 AS max_output_tokens)
);
"""
    )
else:
    _ex(
        f"""
CREATE OR REPLACE TABLE `{P}.{D}.graph_ai_market_dim` AS
SELECT * FROM AI.GENERATE_TEXT(
  MODEL {ML_MODEL},
  (SELECT 'noop' AS prompt WHERE FALSE),
  STRUCT(0.1 AS temperature, 16 AS max_output_tokens)
);
"""
    )

_ex(
    f"""
CREATE OR REPLACE TABLE `{P}.{D}.graph_parsed_market_dim` AS
WITH cleaned AS (
  SELECT
    label AS src_label,
    TRIM(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          COALESCE(
            JSON_VALUE(TO_JSON_STRING(t), '$.ml_generate_text_result'),
            JSON_VALUE(TO_JSON_STRING(t), '$.ml_generate_text_llm_result'),
            JSON_VALUE(TO_JSON_STRING(t), '$.result'),
            ''
          ),
          r'(?s)^```[a-zA-Z0-9]*\\s*',
          ''
        ),
        r'\\s*```$',
        ''
      )
    ) AS txt
  FROM `{P}.{D}.graph_ai_market_dim` AS t
),
parsed AS (
  SELECT
    src_label,
    COALESCE(
      SAFE.PARSE_JSON(txt),
      IF(
        STRPOS(txt, '{{') > 0,
        SAFE.PARSE_JSON(
          SUBSTR(
            txt,
            STRPOS(txt, '{{'),
            LENGTH(txt) - STRPOS(REVERSE(txt), '}}') - STRPOS(txt, '{{') + 2
          )
        ),
        NULL
      )
    ) AS j
  FROM cleaned
  WHERE txt IS NOT NULL AND TRIM(txt) != ''
)
SELECT
  src_label AS market_label,
  NULLIF(TRIM(JSON_VALUE(j, '$.geographic_region')), '') AS geographic_region,
  NULLIF(TRIM(JSON_VALUE(j, '$.product_category')), '') AS product_category
FROM parsed;

CREATE OR REPLACE TABLE `{P}.{D}.edges_in_region` AS
SELECT DISTINCT m.id AS source_node, g.id AS target_node
FROM `{P}.{D}.nodes_market` AS m
JOIN `{P}.{D}.graph_parsed_market_dim` AS p ON p.market_label = m.label
JOIN `{P}.{D}.nodes_geographic_region` AS g ON g.id = p.geographic_region
WHERE p.geographic_region IS NOT NULL;

CREATE OR REPLACE TABLE `{P}.{D}.edges_in_product_category` AS
SELECT DISTINCT m.id AS source_node, c.id AS target_node
FROM `{P}.{D}.nodes_market` AS m
JOIN `{P}.{D}.graph_parsed_market_dim` AS p ON p.market_label = m.label
JOIN `{P}.{D}.nodes_market_category` AS c ON c.id = p.product_category
WHERE p.product_category IS NOT NULL;
"""
)

print("✓ BigQuery graph tables created (staging: graph_staging_*; AI scratch: graph_ai_*).")


## 8. Re-create the Property Graph DDL

Update the `SecGraph` property graph after step 6.3 has created or replaced all node/edge tables in BigQuery (including taxonomy hubs and normalization edges). The cell below still ensures optional taxonomy tables exist, normalizes STRING keys, and runs `06_create_property_graph_ddl.sql`.

In [ ]:
# Ensure optional taxonomy tables exist before creating property graph
fallback_ddls = [
    # Taxonomy node tables
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.nodes_normalized_competitor` (id STRING, label STRING, competitor_type STRING, sector STRING, product_category STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.nodes_geographic_region` (id STRING, label STRING, description STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.nodes_market_category` (id STRING, label STRING, description STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.nodes_risk_category` (id STRING, label STRING, description STRING)",

    # Taxonomy edge tables
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.edges_instance_of` (source_node STRING, target_node STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.edges_subsidiary_of` (source_node STRING, target_node STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.edges_has_risk_category` (source_node STRING, target_node STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.edges_in_region` (source_node STRING, target_node STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.edges_in_product_category` (source_node STRING, target_node STRING)",
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.edges_in_market_category` (source_node STRING, target_node STRING)",

    # Section structure reverse edge guard
    f"CREATE TABLE IF NOT EXISTS `{GCP_PROJECT}.{BQ_DATASET}.edges_section_belongs_to_document` (source_node STRING, target_node STRING)",
]

for ddl in fallback_ddls:
    client.query(ddl, location=BQ_LOCATION).result()

section_belongs_id = f"{GCP_PROJECT}.{BQ_DATASET}.edges_section_belongs_to_document"
try:
    _t = client.get_table(section_belongs_id)
    row_count = list(
        client.query(
            f"SELECT COUNT(1) AS n FROM `{section_belongs_id}`",
            location=BQ_LOCATION,
        ).result()
    )[0]["n"]
except Exception:
    row_count = 0

if int(row_count) == 0:
    client.query(
        f"""
        CREATE OR REPLACE TABLE `{section_belongs_id}` AS
        SELECT DISTINCT target_node AS source_node, source_node AS target_node
        FROM `{GCP_PROJECT}.{BQ_DATASET}.edges_doc_contains_section`
        """,
        location=BQ_LOCATION,
    ).result()
    print("✓ Backfilled edges_section_belongs_to_document from edges_doc_contains_section")

# Normalize key column types to STRING with table rebuild when needed
key_columns = [
    ("nodes_company", ["id"]),
    ("nodes_market", ["id"]),
    ("nodes_risk", ["id"]),
    ("nodes_opportunity", ["id"]),
    ("nodes_competitor", ["id"]),
    ("nodes_reference", ["id"]),
    ("nodes_document", ["id"]),
    ("nodes_section", ["id"]),
    ("nodes_normalized_competitor", ["id"]),
    ("nodes_geographic_region", ["id"]),
    ("nodes_market_category", ["id"]),
    ("nodes_risk_category", ["id"]),
    ("edges_entering", ["source_node", "target_node"]),
    ("edges_expanding", ["source_node", "target_node"]),
    ("edges_exiting", ["source_node", "target_node"]),
    ("edges_faces_risk", ["source_node", "target_node"]),
    ("edges_pursuing", ["source_node", "target_node"]),
    ("edges_competes", ["source_node", "target_node"]),
    ("edges_market_has_reference", ["source_node", "target_node"]),
    ("edges_risk_has_reference", ["source_node", "target_node"]),
    ("edges_opportunity_has_reference", ["source_node", "target_node"]),
    ("edges_competitor_has_reference", ["source_node", "target_node"]),
    ("edges_filed", ["source_node", "target_node"]),
    ("edges_doc_contains_section", ["source_node", "target_node"]),
    ("edges_section_belongs_to_document", ["source_node", "target_node"]),
    ("edges_section_contains_ref", ["source_node", "target_node"]),
    ("edges_instance_of", ["source_node", "target_node"]),
    ("edges_subsidiary_of", ["source_node", "target_node"]),
    ("edges_has_risk_category", ["source_node", "target_node"]),
    ("edges_in_region", ["source_node", "target_node"]),
    ("edges_in_product_category", ["source_node", "target_node"]),
    ("edges_in_market_category", ["source_node", "target_node"]),
]

for tbl, cols in key_columns:
    table_id = f"{GCP_PROJECT}.{BQ_DATASET}.{tbl}"
    try:
        t = client.get_table(table_id)
    except Exception:
        continue

    schema_by_name = {f.name: f.field_type for f in t.schema}
    select_exprs = []
    needs_rebuild = False
    for f in t.schema:
        if f.name in cols and f.field_type != "STRING":
            select_exprs.append(f"CAST(`{f.name}` AS STRING) AS `{f.name}`")
            needs_rebuild = True
        else:
            select_exprs.append(f"`{f.name}`")

    if needs_rebuild:
        rebuild_sql = f"CREATE OR REPLACE TABLE `{table_id}` AS SELECT {', '.join(select_exprs)} FROM `{table_id}`"
        client.query(rebuild_sql, location=BQ_LOCATION).result()
        print(f"Rebuilt {tbl} with STRING key columns")

run_bq_query("06_create_property_graph_ddl.sql")
print("\nPipeline completed successfully!")
